# Escenario 1 — Sin tracking server: file store local

**MLflow setup**

| Pieza | Aquí |
|---|---|
| Tracking server | **no hay** |
| Backend store | sistema de archivos local (`mlruns/`) |
| Artifact store | sistema de archivos local |
| Model Registry | **no disponible** |

**Cuándo tiene sentido:** una persona sola, un experimento exploratorio, una
competencia de Kaggle. Cero infraestructura, cero puertos, cero dependencias.

**Cuándo deja de servir:** en cuanto hay una segunda persona, o hace falta el
Model Registry. Y eso es lo que hay que sentir en este notebook, no leerlo.

> Los notebooks de [`../notebooks/`](../notebooks/) enseñan el **qué** (params,
> métricas, autolog, HPO, registry). Los escenarios enseñan el **dónde**.
>
> El dataset es Iris a propósito: el tema del escenario es la **topología de
> despliegue**, no el modelo. Cambiar de dataset aquí solo añadiría minutos de
> entrenamiento a una discusión de infraestructura.

## Escenario 1: Un cientifico de datos participando de una competición en kaggle

MLflow setup:
* Tracking server: no
* Backend store: sistema de archivos local
* Artifacts store: sistema de archivos local

Usando MLFlow Ui podemos revisar los resultados

In [ ]:
import os

import mlflow
import pandas as pd

En este escenario **no levantamos un Tracking Server**. MLflow guarda todo en un *file store* local.

Para que sea reproducible (y no dependa de rutas absolutas), fijamos el tracking URI a una carpeta `./mlruns` dentro del directorio `scenarios/`.

In [ ]:
tracking_dir = os.path.join(os.getcwd(), "mlruns")
mlflow.set_tracking_uri(f"file://{tracking_dir}")

print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

Define la ubicación donde se almacenarán los metadatos de los experimentos.

En este caso, apunta a la carpeta `mlruns/` dentro de `scenarios/` (file store local).

In [ ]:
mlflow.search_experiments()

### Creemos un experimento y logguemos la corrida

Hagamos una prueba con un dataset de juguete.

#### Definición del Experimento

```python
mlflow.set_experiment("iris-local-no-server")
```

Crea o selecciona un experimento llamado `iris-local-no-server` donde se registrarán todas las ejecuciones relacionadas.

#### Ejecución del Experimento

```python
with mlflow.start_run():
    # Código del experimento
```

El contexto `with mlflow.start_run()` inicia automáticamente una nueva ejecución y la finaliza al salir del bloque.

## Flujo del Experimento

1. Carga de Datos

- Utiliza el dataset de iris de scikit-learn como ejemplo.

2. Definición de Parámetros: 
- C: Parámetro de regularización para LogisticRegression
- random_state: Semilla para reproducibilidad

- mlflow.log_params() registra estos parámetros para futura referencia

3. Entrenamiento del Modelo 

- Entrena un modelo de regresión logística con los parámetros especificados.

4. Evaluación y Logging de Métricas

- Calcula predicciones en el conjunto de entrenamiento
- Registra la métrica de precisión (accuracy) en MLflow

5. Logging del Modelo

- Guarda el modelo entrenado como un artifact, permitiendo su reutilización posterior.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score

mlflow.set_experiment("iris-local-no-server")

with mlflow.start_run(run_name="logreg_baseline"):
    X, y = load_iris(return_X_y=True)

    params = {"C": 0.3, "random_state": 42, "max_iter": 200}
    mlflow.log_params(params)

    mlflow.set_tags(
        {
            "scenario": "1_local_file_store",
            # Pon tu usuario: es el tag que permite responder "quien corrio esto".
            "developer": os.environ.get("USER", "estudiante"),
            "module": "mlops_tracking",
            "model_family": "logistic_regression",
            "dataset": "iris",
        }
    )

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)

    mlflow.log_metric("accuracy", float(accuracy_score(y, y_pred)))
    mlflow.log_metric("f1_macro", float(f1_score(y, y_pred, average="macro")))
    mlflow.log_metric("precision_macro", float(precision_score(y, y_pred, average="macro")))

    # Artifact: predicciones
    preds_path = "predictions_logreg.csv"
    pd.DataFrame({"y_true": y, "y_pred": y_pred}).to_csv(preds_path, index=False)
    mlflow.log_artifact(preds_path)

    # Log del modelo (sin registry en este escenario)
    mlflow.sklearn.log_model(lr, name="model", input_example=X[[0]])

    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

In [ ]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier

mlflow.set_experiment("iris-local-no-server")

with mlflow.start_run(run_name="rf_baseline"):
    X, y = load_iris(return_X_y=True)

    params = {"n_estimators": 200, "random_state": 42, "n_jobs": -1}
    mlflow.log_params(params)

    mlflow.set_tags(
        {
            "scenario": "1_local_file_store",
            # Pon tu usuario: es el tag que permite responder "quien corrio esto".
            "developer": os.environ.get("USER", "estudiante"),
            "module": "mlops_tracking",
            "model_family": "random_forest",
            "dataset": "iris",
        }
    )

    rf = RandomForestClassifier(**params).fit(X, y)
    y_pred = rf.predict(X)

    mlflow.log_metric("accuracy", float(accuracy_score(y, y_pred)))
    mlflow.log_metric("f1_macro", float(f1_score(y, y_pred, average="macro")))

    preds_path = "predictions_rf.csv"
    pd.DataFrame({"y_true": y, "y_pred": y_pred}).to_csv(preds_path, index=False)
    mlflow.log_artifact(preds_path)

    mlflow.sklearn.log_model(rf, name="model", input_example=X[[0]])

    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

## Exploración de Resultados

### Búsqueda de Experimentos

Lista todos los experimentos disponibles en el tracking server.


In [ ]:
mlflow.search_experiments()

Filtrando por nombre del experimento:

In [ ]:
mlflow.search_experiments(filter_string="name = 'iris-local-no-server'")

Experiment ID (obtenido desde el nombre):

In [ ]:
experiment = mlflow.get_experiment_by_name("iris-local-no-server")
assert experiment is not None

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_macro DESC"],
)

runs[[
    "run_id",
    "status",
    "metrics.accuracy",
    "metrics.f1_macro",
    "tags.model_family",
    "tags.scenario",
]]

In [ ]:
# Comparación (ordenando por accuracy)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.accuracy DESC"],
)

runs[[
    "run_id",
    "status",
    "metrics.accuracy",
    "metrics.f1_macro",
    "tags.model_family",
    "tags.scenario",
]]

### Interacción con el Model Registry (limitación en este escenario)

En este escenario *no hay tracking server* (solo file store local).

Por eso, el **Model Registry no es el foco** aquí: lo importante es aprender a loggear params/métricas/tags/artifacts.

En escenarios 2 y 3 veremos el Registry en un entorno más parecido a producción (server + backend DB).

In [ ]:
from mlflow.tracking import MlflowClient
from mlflow.exceptions import MlflowException

client = MlflowClient()

try:
    client.search_registered_models()
except MlflowException:
    print("Model Registry no disponible en este escenario (sin tracking server).")

## Qué acabas de comprobar

El `MlflowException` de la celda anterior no es un error de configuración: es la
**definición** del escenario. Sin backend de base de datos no hay Model Registry,
y sin registry no hay `@champion`, ni versiones, ni rollback por metadatos.

| Limitación | Consecuencia práctica |
|---|---|
| No hay servidor | nadie más puede ver tus runs |
| Backend en archivos | no hay consultas concurrentes fiables |
| Sin registry | el modelo se referencia por ruta o por `run_id`, no por nombre |

Sigue con [`scenario-2-server-local.ipynb`](scenario-2-server-local.ipynb): la
misma API de tracking, un servidor y una base de datos. El código del
entrenamiento **no cambia**; cambia una línea de configuración. Ese es el punto.